In [1]:
import fastplotlib as fpl
import os
import masknmf
import sys
import numpy as np

from ipywidgets import HBox, VBox
import math
import torch

##In the existing folder
import iblwfci_vis
import iblwfci_utils

from tqdm import tqdm
from one.api import ONE
from brainbox.io.one import SessionLoader
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

one = ONE()

%load_ext autoreload

Unable to find extension: VK_EXT_acquire_drm_display
Unable to find extension: VK_EXT_physical_device_drm
No config found!
EGL says it can present to the window but not natively
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA TITAN RTX,DiscreteGPU,Vulkan,555.42.02
❗ limited,"llvmpipe (LLVM 12.0.0, 256 bits)",CPU,Vulkan,Mesa 21.2.6 (LLVM 12.0.0)
❌,NVIDIA TITAN RTX/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 555.42.02


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


In [2]:
"""
The two datasets from FD that we will process are: 
(1) 71ceb3d4-ca68-4380-8fe7-9f63d26222f6 (This is the existing FD_24/2023-06-07/001 data that I had already downloaded) (outputs are in the 007 folder)
(2) 8df7b200-e44c-4c67-82e9-2666ba05d649 (This is the  FD_24/2023-06-08/001 data) (outputs are in the 008 folder)
"""

'\nThe two datasets from FD that we will process are: \n(1) 71ceb3d4-ca68-4380-8fe7-9f63d26222f6 (This is the existing FD_24/2023-06-07/001 data that I had already downloaded) (outputs are in the 007 folder)\n(2) 8df7b200-e44c-4c67-82e9-2666ba05d649 (This is the  FD_24/2023-06-08/001 data) (outputs are in the 008 folder)\n'

In [46]:
eid = "8df7b200-e44c-4c67-82e9-2666ba05d649"
sl = SessionLoader(eid = eid, one=one)
sl.load_trials()
print(sl.trials.keys())
stim_ontimes = sl.trials['stimOn_times']
firstMovement = sl.trials['firstMovement_times'].to_numpy()
firstMovement = firstMovement[~np.isnan(firstMovement)]
feedback_times = sl.trials['feedback_times'].to_numpy()
sl.load_motion_energy() ## Load Motion Energy seems to fail here, not sure why 

sl.load_wheel()
imaging_times = one.load_dataset(eid, "imaging.times.npy", collection="alf/widefield")

/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "2026-01-16", ""
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


Index(['goCueTrigger_times', 'included', 'intervals_bpod_0',
       'intervals_bpod_1', 'quiescencePeriod', 'stimOffTrigger_times',
       'stimOff_times', 'stimOnTrigger_times', 'goCue_times', 'response_times',
       'choice', 'stimOn_times', 'contrastLeft', 'contrastRight',
       'feedback_times', 'feedbackType', 'rewardVolume', 'probabilityLeft',
       'firstMovement_times', 'intervals_0', 'intervals_1'],
      dtype='object')


/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)
/data/home/app2139/wfield/wfieldvenv/lib/python3.11/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2026-01-20"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


In [47]:
## Let's load the 007 folder data (masknmf processing results)
hemocorr_pmd = masknmf.PMDArray.from_hdf5("felicia_jan_26_008/hemocorr.hdf5")
hemocorr_pmd.to('cuda')
hemocorr_pmd.rescale = True

In [48]:
parent_path = "/data/lab/IBL_Alyx/churchlandlab_ucla/Subjects/FD_24/2023-06-08/001/raw_widefield_data"
u_path = os.path.join(parent_path, "U.npy")
svt_path = os.path.join(parent_path, "SVT.npy")
svtcorr_path = os.path.join(parent_path, "SVTcorr.npy")
frames_avg_path = os.path.join(parent_path, "frames_average.npy")

manual_mask = np.load(os.path.join(parent_path, "manual_mask.npy"))

In [49]:
_, _, joao_hemocorr = iblwfci_utils.load_joao_results(u_path,
                  svt_path,
                  svtcorr_path,
                  frames_avg_path,
                  functional_channel = 0)


joao_hemocorr.rescale = True 
joao_hemocorr.to('cuda')

In [50]:
### Code for pulling out wheel/whisker data
def keep_separated(timepoints, min_sep=2.0):
    """
    Keep a subset of timepoints such that each is at least `min_sep`
    seconds apart from the previously kept one.

    Parameters
    ----------
    timepoints : array-like
        List or array of timepoints (in seconds).
    min_sep : float
        Minimum separation in seconds.

    Returns
    -------
    kept : list
        Subset of timepoints satisfying the separation constraint.
    """
    if len(timepoints) == 0:
        return []

    # Ensure sorted order
    timepoints = sorted(timepoints)

    kept = [timepoints[0]]
    last = timepoints[0]

    for t in timepoints[1:]:
        if t - last >= min_sep:
            kept.append(t)
            last = t

    return kept

def load_wheel_speed_trials(wheel_info, imaging_times):
    times = wheel_info['times'].to_numpy()
    velocity = wheel_info['velocity'].to_numpy()

    selector = np.logical_and(times > 5, np.abs(velocity) > 4)
    selector = np.logical_and(selector, times < np.amax(imaging_times) - 100) #Guarantee we are still in the right window
    subset_times = times[selector]
    subset_times = keep_separated(subset_times, min_sep=2.0)
    return np.array(subset_times)


def load_whisker_motion_trials(whisker_motion, imaging_times):
    times = whisker_motion['times'].to_numpy()
    energy = whisker_motion['whiskerMotionEnergy'].to_numpy()
    print(np.amax(energy))

    selector = np.logical_and(times > 5, np.abs(energy) > 10)
    selector = np.logical_and(selector, times < np.amax(imaging_times) - 100)
    subset_times = times[selector]
    subset_times = keep_separated(subset_times, min_sep=2.0)
    return np.array(subset_times)

wheel_trials = load_wheel_speed_trials(sl.wheel, imaging_times)
whisker_trials = load_whisker_motion_trials(sl.motion_energy['leftCamera'], imaging_times)
print(len(whisker_trials))
print(len(wheel_trials))

49.949092864990234
1460
328


In [51]:


def get_trial_triggered_stack_binary_xcorr(my_pmd_movie, trial_indices, mean_img, before_frames = 20, after_frames = 100, device='cpu', normalizer = True):
    my_pmd_movie.to(device)
    average_movie = torch.zeros((before_frames + after_frames, my_pmd_movie.shape[1], my_pmd_movie.shape[2])).to(device)
    num_trials = trial_indices.shape[0]
    if normalizer is not None:
        normalizer = torch.from_numpy(normalizer).to(device)
        normalizer[normalizer == 0] = 1.0 
    else:
        normalizer = torch.ones(my_pmd_movie.shape[1], my_pmd_movie.shape[2], device='cuda').float()
    for k in tqdm(range(trial_indices.shape[0])):
        curr_ind = trial_indices[k]
        curr_data = my_pmd_movie.getitem_tensor(slice(curr_ind - before_frames, curr_ind + after_frames)) 
        curr_data /= my_pmd_movie.shape[0]
        average_movie += curr_data 
    numerator = average_movie - (num_trials / my_pmd_movie.shape[0])*torch.from_numpy(mean_img).to(device).float()
    numerator = torch.nan_to_num(numerator / normalizer[None, :, :], nan = 0.0)
    test_vec = torch.zeros(my_pmd_movie.shape[0], device = 'cuda').float()
    test_vec[:num_trials] = 1
    std_val = torch.std(test_vec)
    print(std_val)
    numerator = torch.nan_to_num(numerator / std_val, nan = 0.0)
    return numerator.cpu().numpy()



def compute_std_img(pmd_obj, device = 'cuda', batch_size = 40):
    
    height_iters = math.ceil(pmd_obj.shape[1] / batch_size)
    width_iters = math.ceil(pmd_obj.shape[2] / batch_size)

    std_map = torch.zeros(pmd_obj.shape[1], pmd_obj.shape[2]).to(device)
    mean_map = torch.zeros(pmd_obj.shape[1], pmd_obj.shape[2]).to(device)
    for k in tqdm(range(height_iters)):
        for j in range(width_iters):
            dim1 = slice(k * batch_size, min((k+1)*batch_size, pmd_obj.shape[1]))
            dim2 = slice(j * batch_size, min((j+1)*batch_size, pmd_obj.shape[2]))
            my_data = pmd_obj.getitem_tensor((slice(0, pmd_obj.shape[0]), dim1, dim2))
            std_map[dim1, dim2] = torch.std(my_data, dim = 0)
            mean_map[dim1, dim2] = torch.mean(my_data, dim = 0)
        
    return std_map.cpu().numpy(), mean_map.cpu().numpy()

In [57]:
trial_indices_abs = np.searchsorted(imaging_times, whisker_trials)
trial_indices = (trial_indices_abs // 2) ##This is correct if the functional channel is 0. If it is 1, you need to adjust it
before_frames = 20
after_frames = 40


In [53]:
joao_hemo_std_img, joao_hemo_mean_img = compute_std_img(joao_hemocorr)
amol_hemo_std_img, amol_hemo_mean_img = compute_std_img(hemocorr_pmd)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 14/14 [00:14<00:00,  1.02s/it]


In [58]:
amol_hemo_trial_avg =  get_trial_triggered_stack_binary_xcorr(hemocorr_pmd, 
                                                              trial_indices, 
                                                              amol_hemo_mean_img, 
                                                              before_frames = before_frames, 
                                                              after_frames = after_frames, 
                                                              device='cuda', 
                                                              normalizer=amol_hemo_std_img)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1460/1460 [00:11<00:00, 122.10it/s]


tensor(0.1131, device='cuda:0')


In [59]:
joao_hemo_trial_avg =  get_trial_triggered_stack_binary_xcorr(joao_hemocorr, 
                                                         trial_indices, 
                                                         joao_hemo_mean_img, 
                                                         before_frames = before_frames, 
                                                         after_frames = after_frames, 
                                                         device='cuda', 
                                                         normalizer=joao_hemo_std_img)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1460/1460 [01:11<00:00, 20.46it/s]


tensor(0.1131, device='cuda:0')


In [61]:
iw = fpl.ImageWidget(data = [manual_mask * (amol_hemo_trial_avg),
                             manual_mask*(joao_hemo_trial_avg)],
                     names = ['amol hemo whisker triggered',
                              'joao hemo whisker triggered'],
                     figure_shape = (1, 2),
                    histogram_widget=True)

iw.managed_graphics[0].vmax = 0.10
iw.managed_graphics[1].vmax = 0.10
iw.managed_graphics[0].vmin = -0.05
iw.managed_graphics[1].vmin = -0.05

iw.cmap = "gray"
iw.show()


In [44]:

def plot_images_with_bbox_and_timeseries(
    frame,
    img_left,
    img_right,
    bbox,  # (x, y, width, height)
    name_save = None
):

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], hspace=0.3)

    ax_img_left  = fig.add_subplot(gs[0, 0])
    ax_img_right = fig.add_subplot(gs[0, 1])
    ax_ts_left   = fig.add_subplot(gs[1, 0])
    ax_ts_right  = fig.add_subplot(gs[1, 1])

    # ---------------- Images ----------------
    vmin, vmax = -0.05, 0.1

    im0 = ax_img_left.imshow(img_left[frame], cmap="gray", vmin=vmin, vmax=vmax)
    ax_img_left.set_title("PMD XCorr Wheel Velocity Onset")
    ax_img_left.axis("off")

    im1 = ax_img_right.imshow(img_right[frame], cmap="gray", vmin=vmin, vmax=vmax)
    ax_img_right.set_title("WField XCorr Wheel Velocity")
    ax_img_right.axis("off")

    # ----------- Crosshair Location -----------
    x, y, w, h = bbox
    cx = x + w // 2
    cy = y + h // 2

    # Draw crosshairs (full image span)
    for ax, img in zip(
        [ax_img_left, ax_img_right],
        [img_left[frame], img_right[frame]]
    ):
        height, width = img.shape

        ax.axvline(cx, color="red", linewidth=0.5, linestyle='dotted')
        ax.axhline(cy, color="red", linewidth=0.5, linestyle='dotted')

    # Shared colorbar
    cbar = fig.colorbar(im0, ax=[ax_img_left, ax_img_right])
    cbar.set_label("Value")

    # ---------------- Time Series ----------------
    ax_ts_left.plot(
        np.mean(img_left[:, y:y+h, x:x+w], axis=(1, 2)),
        color="red"
    )
    ax_ts_left.set_title(f"PMD XCorr at center ({cy}, {cx})")
    ax_ts_left.set_xlabel("Time")
    ax_ts_left.set_ylabel("Signal")
    ax_ts_left.grid(alpha=0.3)

    ax_ts_right.plot(
        np.mean(img_right[:, y:y+h, x:x+w], axis=(1, 2)),
        color="red"
    )
    ax_ts_right.set_title(f"wfield XCorr at center ({cy}, {cx})")
    ax_ts_right.set_xlabel("Time")
    ax_ts_right.set_ylabel("Signal")
    ax_ts_right.grid(alpha=0.3)

    plt.tight_layout()
    if name_save is not None:
        plt.savefig(name_save, bbox_inches = "tight")
    plt.show()


def plot_images_with_bbox_and_timeseries(
    frame,
    img_left,
    img_right,
    bbox,  # (x, y, width, height)
    name_save=None
):

    fig = plt.figure(figsize=(12, 8))
    gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], hspace=0.3)

    ax_img_left  = fig.add_subplot(gs[0, 0])
    ax_img_right = fig.add_subplot(gs[0, 1])
    ax_ts_left   = fig.add_subplot(gs[1, 0])
    ax_ts_right  = fig.add_subplot(gs[1, 1])

    # ---------------- Images ----------------
    vmin, vmax = -0.05, 0.1

    im0 = ax_img_left.imshow(img_left[frame], cmap="gray", vmin=vmin, vmax=vmax)
    ax_img_left.set_title("PMD XCorr Wheel Velocity Onset")
    ax_img_left.axis("off")

    im1 = ax_img_right.imshow(img_right[frame], cmap="gray", vmin=vmin, vmax=vmax)
    ax_img_right.set_title("WField XCorr Wheel Velocity")
    ax_img_right.axis("off")

    # ----------- Crosshair Location -----------
    x, y, w, h = bbox
    cx = x + w // 2
    cy = y + h // 2

    for ax, img in zip(
        [ax_img_left, ax_img_right],
        [img_left[frame], img_right[frame]]
    ):
        ax.axvline(cx, color="red", linewidth=0.5, linestyle="dotted")
        ax.axhline(cy, color="red", linewidth=0.5, linestyle="dotted")

    # Shared colorbar
    cbar = fig.colorbar(im0, ax=[ax_img_left, ax_img_right])
    cbar.set_label("Cross-correlation")

    # ---------------- Time Series (Cross-Correlation) ----------------
    trace_left = np.mean(img_left[:, y:y+h, x:x+w], axis=(1, 2))
    trace_right = np.mean(img_right[:, y:y+h, x:x+w], axis=(1, 2))

    n_lags = len(trace_left)
    zero_lag_index = 20  # given
    lags = np.arange(n_lags) - zero_lag_index

    # ---- Left trace ----
    ax_ts_left.plot(lags, trace_left, color="red")
    ax_ts_left.axvline(0, linestyle="--", linewidth=1)  # zero lag marker
    ax_ts_left.set_title(f"PMD Cross-Correlation @ ({cy}, {cx})")
    ax_ts_left.set_xlabel("Lag")
    ax_ts_left.set_ylabel("Correlation")
    ax_ts_left.grid(alpha=0.3)

    # ---- Right trace ----
    ax_ts_right.plot(lags, trace_right, color="red")
    ax_ts_right.axvline(0, linestyle="--", linewidth=1)
    ax_ts_right.set_title(f"WField Cross-Correlation @ ({cy}, {cx})")
    ax_ts_right.set_xlabel("Lag")
    ax_ts_right.set_ylabel("Correlation")
    ax_ts_right.grid(alpha=0.3)

    plt.tight_layout()

    if name_save is not None:
        plt.savefig(name_save, bbox_inches="tight")

    plt.show()

In [62]:
plot_images_with_bbox_and_timeseries(
    36,
    manual_mask * (amol_hemo_trial_avg),
    manual_mask*(joao_hemo_trial_avg),
    (340, 300, 1, 1),  # (x, y, width, height)
    name_save = "wheel_vel_xcorr_007.png"
)